In [0]:
DELETE FROM workspace.treinamentos.silver_celulares_sp
WHERE LATITUDE <= -100 AND LONGITUDE <= -100

In [0]:
DELETE FROM workspace.treinamentos.silver_celulares_sp
WHERE DATA_OCORRENCIA_BO < DATE '2020-01-01'

In [0]:
DELETE FROM workspace.treinamentos.silver_celulares_sp
WHERE QUANTIDADE_OBJETO > 100

In [0]:
CREATE OR REPLACE TABLE workspace.treinamentos.silver_celulares_sp AS
SELECT DISTINCT * FROM workspace.treinamentos.silver_celulares_sp

In [0]:
MERGE INTO workspace.treinamentos.silver_celulares_sp t
USING (
  WITH refs_bairro AS (
    SELECT BAIRRO, CIDADE,
           AVG(LATITUDE) AS LATITUDE_REF, AVG(LONGITUDE) AS LONGITUDE_REF
    FROM workspace.treinamentos.silver_celulares_sp
    WHERE LATITUDE IS NOT NULL AND LONGITUDE IS NOT NULL AND LATITUDE != 0 AND LONGITUDE != 0
    GROUP BY BAIRRO, CIDADE
  ),
  refs_cidade AS (
    SELECT CIDADE,
           AVG(LATITUDE) AS LATITUDE_REF, AVG(LONGITUDE) AS LONGITUDE_REF
    FROM workspace.treinamentos.silver_celulares_sp
    WHERE LATITUDE IS NOT NULL AND LONGITUDE IS NOT NULL AND LATITUDE != 0 AND LONGITUDE != 0
    GROUP BY CIDADE
  ),
  pares AS (
    SELECT DISTINCT COALESCE(BAIRRO, '') AS BAIRRO_KEY, BAIRRO, CIDADE
    FROM workspace.treinamentos.silver_celulares_sp
  )
  SELECT p.BAIRRO_KEY, p.CIDADE,
         COALESCE(b.LATITUDE_REF, c.LATITUDE_REF) AS LATITUDE_REF,
         COALESCE(b.LONGITUDE_REF, c.LONGITUDE_REF) AS LONGITUDE_REF
  FROM pares p
  LEFT JOIN refs_bairro b ON b.BAIRRO = p.BAIRRO AND b.CIDADE = p.CIDADE
  LEFT JOIN refs_cidade c ON c.CIDADE = p.CIDADE
) s
ON s.BAIRRO_KEY = COALESCE(t.BAIRRO, '') AND s.CIDADE = t.CIDADE
WHEN MATCHED AND s.LATITUDE_REF IS NOT NULL
  AND (t.LATITUDE IS NULL OR t.LONGITUDE IS NULL OR t.LATITUDE = 0 OR t.LONGITUDE = 0) THEN
  UPDATE SET t.LATITUDE = s.LATITUDE_REF, t.LONGITUDE = s.LONGITUDE_REF

In [0]:
UPDATE workspace.treinamentos.silver_celulares_sp
SET LATITUDE = NULL, LONGITUDE = NULL
WHERE LATITUDE IS NULL OR LONGITUDE IS NULL OR LATITUDE = 0 OR LONGITUDE = 0

In [0]:
SELECT * FROM workspace.treinamentos.silver_celulares_sp
where LATITUDE is not null and LONGITUDE is not NULL

In [0]:
select count(CEP) from workspace.treinamentos.silver_celulares_sp
where cep is not null


In [0]:
Update workspace.treinamentos.silver_celulares_sp
set CEP = LEFT(CEP, length(CEP)-1)
where length(CEP) > 8


In [0]:
MERGE INTO workspace.treinamentos.silver_celulares_sp t
USING (
WITH validos AS (
  SELECT CIDADE, LOGRADOURO, BAIRRO, CEP, COUNT(*) AS QTD
  FROM workspace.treinamentos.silver_celulares_sp
  WHERE CEP RLIKE '^[0-9]{8}$' AND CEP != '00000000'
  GROUP BY 1, 2, 3, 4
),
moda_rua AS (
  SELECT CIDADE, LOGRADOURO, CEP AS MODA
  FROM (SELECT CIDADE, LOGRADOURO, CEP, SUM(QTD) AS QTD FROM validos WHERE LOGRADOURO IS NOT NULL GROUP BY 1, 2, 3)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY CIDADE, LOGRADOURO ORDER BY QTD DESC, CEP) = 1
),
moda_bairro AS (
  SELECT CIDADE, BAIRRO, CEP AS MODA
  FROM (SELECT CIDADE, BAIRRO, CEP, SUM(QTD) AS QTD FROM validos WHERE BAIRRO IS NOT NULL GROUP BY 1, 2, 3)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY CIDADE, BAIRRO ORDER BY QTD DESC, CEP) = 1
),
moda_cidade AS (
  SELECT CIDADE, CEP AS MODA
  FROM (SELECT CIDADE, CEP, SUM(QTD) AS QTD FROM validos GROUP BY 1, 2)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY CIDADE ORDER BY QTD DESC, CEP) = 1
),
pares AS (
  SELECT DISTINCT CIDADE, COALESCE(LOGRADOURO, '') AS RUA_KEY, COALESCE(BAIRRO, '') AS BAIRRO_KEY
  FROM workspace.treinamentos.silver_celulares_sp
)
SELECT p.CIDADE, p.RUA_KEY, p.BAIRRO_KEY,
       COALESCE(r.MODA, b.MODA, c.MODA) AS CEP_NOVO
FROM pares p
LEFT JOIN moda_rua r ON r.CIDADE = p.CIDADE AND r.LOGRADOURO = p.RUA_KEY
LEFT JOIN moda_bairro b ON b.CIDADE = p.CIDADE AND b.BAIRRO = p.BAIRRO_KEY
LEFT JOIN moda_cidade c ON c.CIDADE = p.CIDADE
) s
ON t.CIDADE = s.CIDADE
  AND COALESCE(t.LOGRADOURO, '') = s.RUA_KEY
  AND COALESCE(t.BAIRRO, '') = s.BAIRRO_KEY
WHEN MATCHED AND s.CEP_NOVO IS NOT NULL
  AND (t.CEP IS NULL OR NOT (t.CEP RLIKE '^[0-9]{8}$' AND t.CEP != '00000000')) THEN
  UPDATE SET t.CEP = s.CEP_NOVO


In [0]:
select CIDADE, BAIRRO, CEP, LATITUDE, LONGITUDE from workspace.treinamentos.celulares_bronze
where CIDADE in  ('CAMPINA GRANDE SUL','COLOMBO')

In [0]:
select CIDADE, BAIRRO, LOGRADOURO, (distinct (CEP)) from workspace.treinamentos.silver_celulares_sp
